# Polymarket Investment Adviser — Daily Signals

Visualización diaria de señales de inversión generadas por el pipeline.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime, timedelta
from IPython.display import display

from src.scraper import PolymarketClient, TrendingDetector
from src.correlator import StockCorrelator
from src.predictor import SignalGenerator
from src.utils import config, logger

import logging
logging.getLogger('polymarket').setLevel(logging.WARNING)

print(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('Setup OK')

## 1. Correr Pipeline

In [ ]:
# --- Step 1: Scrape
client = PolymarketClient()
markets = client.get_trending_markets(limit=100)
client.save_snapshot(markets)

# --- Step 2: Detect trending signals
detector = TrendingDetector(client)
trending_signals = detector.detect_all(markets)

# --- Step 3: Correlate with stocks
correlator = StockCorrelator()
stock_signals = correlator.correlate(trending_signals)

# --- Step 4: Generate investment signals
generator = SignalGenerator()
inv_signals = generator.generate(stock_signals)

print(f'Mercados descargados: {len(markets)}')
print(f'Señales trending:     {len(trending_signals)}')
print(f'Correlaciones stock:  {len(stock_signals)}')
print(f'Señales inversión:    {len(inv_signals)}')

## 2. Top Mercados por Volumen 24h

In [ ]:
df_markets = pd.DataFrame([
    {
        'question': m.question[:65],
        'yes_price': m.outcome_prices.get('Yes', None),
        'volume_24h': m.volume_24h,
        'volume_total': m.volume_total,
        'liquidity': m.liquidity,
        'end_date': m.end_date,
        'token_id': None  # filled below
    }
    for m in markets
])

# Attach raw token IDs for price history later
_raw = client.get_markets(limit=100)
_id_map = {}
for r in _raw:
    try:
        tids = json.loads(r.get('clobTokenIds', '[]'))
        _id_map[r.get('question', '')[:65]] = tids[0] if tids else None
    except Exception:
        pass
df_markets['token_id'] = df_markets['question'].map(_id_map)

top20 = df_markets.nlargest(20, 'volume_24h').copy()

fig, ax = plt.subplots(figsize=(14, 8))

colors = ['#2ecc71' if (p is not None and p > 0.5) else '#e74c3c' if (p is not None and p < 0.2) else '#95a5a6'
          for p in top20['yes_price']]

bars = ax.barh(range(len(top20)), top20['volume_24h'] / 1e6, color=colors, edgecolor='none', height=0.7)

ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['question'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Volumen 24h (millones USD)', fontsize=11)
ax.set_title(f'Top 20 Mercados Polymarket por Volumen 24h\n{datetime.now().strftime("%Y-%m-%d")}', fontsize=13, fontweight='bold')

# YES price labels
for i, (_, row) in enumerate(top20.iterrows()):
    p = row['yes_price']
    label = f'YES: {p:.0%}' if p is not None else 'N/A'
    ax.text(row['volume_24h']/1e6 + 0.02, i, label, va='center', fontsize=8, color='#2c3e50')

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='YES > 50% (probable)'),
    mpatches.Patch(color='#e74c3c', label='YES < 20% (improbable)'),
    mpatches.Patch(color='#95a5a6', label='Incierto (20-50%)'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/chart_top_markets.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: data/processed/chart_top_markets.png')

## 3. Señales de Inversión

In [ ]:
df_signals = pd.DataFrame([
    {
        'ticker': s.ticker,
        'name': s.instrument_name,
        'type': s.instrument_type,
        'action': s.action,
        'strength': s.strength,
        'confidence': s.confidence,
        'timing': s.timing_action,
        'yes_price': s.yes_price,
        'volume_24h': s.volume_24h,
        'source': s.source_market[:60],
        'category': s.source_category,
    }
    for s in inv_signals
]).sort_values('confidence', ascending=False)

# --- Breakdown donut chart
action_counts = df_signals['action'].value_counts()
action_colors = {'BUY': '#2ecc71', 'SELL': '#e74c3c', 'WATCH': '#f39c12', 'HOLD': '#95a5a6'}
colors_pie = [action_colors.get(a, '#bdc3c7') for a in action_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Donut
wedges, texts, autotexts = ax1.pie(
    action_counts.values,
    labels=action_counts.index,
    colors=colors_pie,
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops=dict(width=0.5),
    textprops={'fontsize': 12}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax1.set_title('Distribución de Señales', fontsize=12, fontweight='bold')
ax1.text(0, 0, f'{len(df_signals)}\nseñales', ha='center', va='center', fontsize=14, fontweight='bold')

# Confidence bar por señal
colors_bar = [action_colors.get(a, '#bdc3c7') for a in df_signals['action']]
y_pos = range(len(df_signals))
bars2 = ax2.barh(y_pos, df_signals['confidence'], color=colors_bar, edgecolor='none', height=0.65)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f"{row['action']:5} {row['ticker']}" for _, row in df_signals.iterrows()], fontsize=9)
ax2.invert_yaxis()
ax2.set_xlim(0, 1.15)
ax2.set_xlabel('Confianza', fontsize=11)
ax2.set_title('Confianza por Señal', fontsize=12, fontweight='bold')
ax2.axvline(0.75, color='#2c3e50', linestyle='--', alpha=0.4, label='Umbral 75%')
ax2.legend(fontsize=9)
for i, (_, row) in enumerate(df_signals.iterrows()):
    ax2.text(row['confidence'] + 0.01, i, f"{row['confidence']:.0%}", va='center', fontsize=8)
ax2.grid(axis='x', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle(f'Investment Signals — {datetime.now().strftime("%Y-%m-%d %H:%M")}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/chart_signals.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Historial de Precios — Top Mercados (CLOB API)

In [ ]:
def fetch_price_history(token_id: str, fidelity: int = 60) -> list:
    """Fetch price history from Polymarket CLOB API."""
    url = f'https://clob.polymarket.com/prices-history?interval=1d&market={token_id}&fidelity={fidelity}'
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json().get('history', [])
    except Exception:
        pass
    return []

# Fetch history for top 12 markets by volume that have a token_id
top_with_tokens = top20[top20['token_id'].notna()].head(12)

histories = {}
for _, row in top_with_tokens.iterrows():
    h = fetch_price_history(row['token_id'])
    if h:
        histories[row['question']] = h
        print(f"  OK  {len(h):3} pts  YES: {row['yes_price']:.0%}  | {row['question'][:65]}")
    else:
        print(f"  --  sin datos                  | {row['question'][:65]}")

print(f'\nHistorial obtenido para {len(histories)} mercados')

In [ ]:
if not histories:
    print('Sin datos de historial disponibles')
else:
    n = len(histories)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
    axes = axes.flatten() if n > 1 else [axes]

    for ax, (question, history) in zip(axes, histories.items()):
        ts = [datetime.utcfromtimestamp(p['t']) for p in history]
        prices = [p['p'] for p in history]

        # Color based on trend
        if len(prices) >= 2:
            trend_color = '#2ecc71' if prices[-1] >= prices[0] else '#e74c3c'
        else:
            trend_color = '#95a5a6'

        ax.plot(ts, prices, color=trend_color, linewidth=2)
        ax.fill_between(ts, prices, alpha=0.15, color=trend_color)
        ax.axhline(0.5, color='#7f8c8d', linestyle='--', alpha=0.4, linewidth=0.8)

        # Current price annotation
        if prices:
            ax.annotate(
                f'NOW: {prices[-1]:.0%}',
                xy=(ts[-1], prices[-1]),
                xytext=(-45, 8), textcoords='offset points',
                fontsize=8, fontweight='bold', color=trend_color
            )

        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel('Prob YES', fontsize=8)
        ax.set_title(question[:55] + ('...' if len(question) > 55 else ''), fontsize=8, fontweight='bold')
        ax.tick_params(axis='x', labelsize=7, rotation=30)
        ax.tick_params(axis='y', labelsize=8)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax.grid(alpha=0.2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Hide unused subplots
    for ax in axes[len(histories):]:
        ax.set_visible(False)

    plt.suptitle(f'Historial de Precios — Últimas 24h (CLOB API)\n{datetime.now().strftime("%Y-%m-%d %H:%M")}',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../data/processed/chart_price_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado: data/processed/chart_price_history.png')

## 5. Tabla Detallada de Señales

In [ ]:
def color_action(val):
    if val == 'BUY':  return 'background-color: #d5f5e3; color: #1e8449; font-weight: bold'
    if val == 'SELL': return 'background-color: #fadbd8; color: #922b21; font-weight: bold'
    return 'background-color: #fef9e7; color: #7d6608'

def color_timing(val):
    if val == 'act_now': return 'background-color: #d5f5e3; font-weight: bold'
    if val == 'prepare': return 'background-color: #d6eaf8'
    if val == 'late':    return 'background-color: #fdebd0'
    return ''

def fmt_pct(val):
    try: return f'{float(val):.0%}'
    except: return val

display_df = df_signals[['ticker','name','action','strength','confidence','timing','yes_price','volume_24h','source']].copy()
display_df['confidence'] = display_df['confidence'].map(fmt_pct)
display_df['yes_price']  = display_df['yes_price'].map(fmt_pct)
display_df['volume_24h'] = display_df['volume_24h'].map(lambda x: f'${x:,.0f}')
display_df.columns = ['Ticker','Nombre','Acción','Fuerza','Confianza','Timing','YES%','Vol 24h','Fuente (mercado)']

styled = display_df.style \
    .applymap(color_action, subset=['Acción']) \
    .applymap(color_timing, subset=['Timing']) \
    .set_properties(**{'font-size': '11px'}) \
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '11px')]}])

display(styled)

## 6. Momentum: Volumen 24h vs Volumen Semanal

In [ ]:
# Fetch raw markets with weekly volume for momentum analysis
_raw_markets = client.get_markets(limit=50)

df_momentum = pd.DataFrame([
    {
        'question': r.get('question', '')[:55],
        'vol_24h': float(r.get('volume24hr', 0) or 0),
        'vol_1wk': float(r.get('volume1wk', 0) or 0),
        'yes_price': float(json.loads(r.get('outcomePrices', '[0]'))[0]) if r.get('outcomePrices') else 0,
    }
    for r in _raw_markets
])

# avg daily vol this week vs today
df_momentum = df_momentum[df_momentum['vol_1wk'] > 0].copy()
df_momentum['avg_daily_wk'] = df_momentum['vol_1wk'] / 7
df_momentum['momentum_ratio'] = df_momentum['vol_24h'] / df_momentum['avg_daily_wk']
df_momentum = df_momentum[df_momentum['vol_24h'] > 50000].nlargest(20, 'momentum_ratio')

fig, ax = plt.subplots(figsize=(14, 7))

bar_colors = ['#e74c3c' if r > 3 else '#f39c12' if r > 1.5 else '#95a5a6'
              for r in df_momentum['momentum_ratio']]

bars = ax.barh(range(len(df_momentum)), df_momentum['momentum_ratio'], color=bar_colors, height=0.7)
ax.set_yticks(range(len(df_momentum)))
ax.set_yticklabels(df_momentum['question'], fontsize=9)
ax.invert_yaxis()
ax.axvline(1.0, color='#2c3e50', linestyle='--', alpha=0.5, label='Promedio semanal (1x)')
ax.axvline(3.0, color='#e74c3c', linestyle='--', alpha=0.4, label='Spike (3x)')
ax.set_xlabel('Vol 24h / Promedio diario semanal (momentum ratio)', fontsize=11)
ax.set_title(f'Momentum de Volumen — Actividad Hoy vs Promedio Semanal\n{datetime.now().strftime("%Y-%m-%d")}',
             fontsize=12, fontweight='bold')

for i, (_, row) in enumerate(df_momentum.iterrows()):
    ax.text(row['momentum_ratio'] + 0.05, i, f"{row['momentum_ratio']:.1f}x", va='center', fontsize=8, color='#2c3e50')

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='Spike fuerte (>3x)'),
    mpatches.Patch(color='#f39c12', label='Spike moderado (1.5-3x)'),
    mpatches.Patch(color='#95a5a6', label='Normal (<1.5x)'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/chart_momentum.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: data/processed/chart_momentum.png')